# Session 3: LangGraph Orchestration & Workflow Design (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 3. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks.

**McKinsey Context:** In this session we model consulting workflows as directed graphs -- from engagement scoping through analysis to partner-ready deliverables. Every demo and task uses realistic McKinsey scenarios: PE due diligence pipelines, client onboarding, market entry assessments, and iterative recommendation refinement.

## Learning Objectives

By the end of this session, students will be able to:

1. **Create StateGraphs** with typed state schemas to model consulting engagement pipelines
2. **Define nodes and edges** to represent workflow steps such as data gathering, market analysis, and recommendation formulation
3. **Add conditional edges** for dynamic routing based on consulting decisions (engagement complexity, client tier, industry sector)
4. **Build cyclic workflows** for iterative refinement until deliverables reach partner-quality standards
5. **Implement a ReAct agent** as a McKinsey analyst researching markets, analyzing financials, and benchmarking competitors
6. **Add human-in-the-loop** checkpoints for partner review and client sign-off

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are identical to the student notebook. Each demo uses McKinsey consulting scenarios to illustrate LangGraph concepts.

### Demo 1: LangGraph Basics -- Modeling a Consulting Engagement Pipeline

We model a simple engagement intake pipeline: a new client request arrives, we extract the engagement scope, classify the industry sector, and format a preliminary engagement summary. This demonstrates the core StateGraph components -- typed state, nodes, and linear edges.

In [ ]:
# Demo 1 - LangGraph Basics: Consulting Engagement Pipeline

class EngagementState(TypedDict):
    client_request: str
    scope_summary: str
    industry_sector: str
    engagement_brief: str

def extract_scope_node(state: EngagementState) -> dict:
    """Extract and normalize the engagement scope from the client request."""
    return {"scope_summary": state["client_request"].strip().upper()}

def classify_industry_node(state: EngagementState) -> dict:
    """Classify the industry sector based on keywords in the request."""
    text = state["client_request"].lower()
    if any(kw in text for kw in ["bank", "financial", "fintech", "insurance"]):
        sector = "Financial Services"
    elif any(kw in text for kw in ["pharma", "health", "biotech", "hospital"]):
        sector = "Healthcare & Life Sciences"
    elif any(kw in text for kw in ["retail", "consumer", "e-commerce", "cpg"]):
        sector = "Consumer & Retail"
    else:
        sector = "Cross-Industry"
    return {"industry_sector": sector}

def format_brief_node(state: EngagementState) -> dict:
    """Format the preliminary engagement brief."""
    return {"engagement_brief": f"ENGAGEMENT BRIEF\nScope: {state['scope_summary']}\nSector: {state['industry_sector']}"}

graph = StateGraph(EngagementState)
graph.add_node("extract_scope", extract_scope_node)
graph.add_node("classify_industry", classify_industry_node)
graph.add_node("format_brief", format_brief_node)
graph.set_entry_point("extract_scope")
graph.add_edge("extract_scope", "classify_industry")
graph.add_edge("classify_industry", "format_brief")
graph.add_edge("format_brief", END)

app = graph.compile()
result = app.invoke({
    "client_request": "Our fintech startup needs help with market entry strategy for Southeast Asia",
    "scope_summary": "", "industry_sector": "", "engagement_brief": ""
})
print(result["engagement_brief"])

### Demo 2: Conditional Edges -- Routing Client Engagements by Complexity

In consulting, the workflow changes based on engagement characteristics. A quick benchmarking study follows a different path than a full-scale transformation program. Here we use an LLM to classify engagement complexity and route to the appropriate workstream: **rapid assessment**, **standard engagement**, or **transformation program**.

In [ ]:
# Demo 2 - Conditional Edges: Routing Engagements by Complexity

class EngagementRouterState(TypedDict):
    engagement_request: str
    complexity: str
    workstream_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def assess_complexity_node(state: EngagementRouterState) -> dict:
    """Use LLM to classify engagement complexity."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Classify this client request into exactly one category: 'rapid' (2-4 week benchmarking or quick scan), 'standard' (8-12 week strategy or operations engagement), or 'transformation' (6+ month large-scale change program). Respond with just the one word."),
        HumanMessage(content=state["engagement_request"])
    ])
    complexity = response.content.strip().lower()
    print(f"  Complexity assessed: {complexity}")
    return {"complexity": complexity}

def rapid_assessment_node(state: EngagementRouterState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey associate conducting a rapid assessment. Provide a concise 2-week diagnostic framework with key hypotheses to test."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[RAPID ASSESSMENT]\n{r.content}"}

def standard_engagement_node(state: EngagementRouterState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager designing a standard 10-week engagement. Outline the workplan with key phases: diagnostic, analysis, recommendations, and implementation roadmap."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[STANDARD ENGAGEMENT]\n{r.content}"}

def transformation_node(state: EngagementRouterState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner designing a large-scale transformation program. Outline the multi-phase approach including change management, capability building, and performance tracking."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[TRANSFORMATION PROGRAM]\n{r.content}"}

def route_by_complexity(state: EngagementRouterState) -> str:
    c = state["complexity"]
    if "rapid" in c: return "rapid_assessment"
    elif "transformation" in c: return "transformation"
    return "standard_engagement"

graph = StateGraph(EngagementRouterState)
graph.add_node("assess_complexity", assess_complexity_node)
graph.add_node("rapid_assessment", rapid_assessment_node)
graph.add_node("standard_engagement", standard_engagement_node)
graph.add_node("transformation", transformation_node)
graph.set_entry_point("assess_complexity")
graph.add_conditional_edges("assess_complexity", route_by_complexity, {
    "rapid_assessment": "rapid_assessment",
    "standard_engagement": "standard_engagement",
    "transformation": "transformation"
})
graph.add_edge("rapid_assessment", END)
graph.add_edge("standard_engagement", END)
graph.add_edge("transformation", END)
app = graph.compile()

for req in [
    "We need a quick competitive benchmarking of our pricing vs. top 3 rivals",
    "Help us design a new go-to-market strategy for entering the European healthcare market",
    "We are a $20B conglomerate and need to completely restructure our operating model across 15 business units"
]:
    r = app.invoke({"engagement_request": req, "complexity": "", "workstream_output": ""})
    print(f"\nRequest: {req}")
    print(f"Output: {r['workstream_output'][:200]}...\n{'='*60}")

### Demo 3: Building a ReAct Agent -- McKinsey Market Research Analyst

The ReAct (Reason + Act) pattern models how a McKinsey analyst works: reason about what data is needed, search for market intelligence, observe the findings, and iterate until a well-supported answer emerges. Our analyst has access to simulated tools for market data, financial metrics, and competitive intelligence.

In [ ]:
# Demo 3 - ReAct Agent: McKinsey Market Research Analyst

class ReActState(TypedDict):
    question: str
    thoughts: list[str]
    actions: list[str]
    observations: list[str]
    final_answer: str
    step_count: int

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Simulated McKinsey research tools
def research_tool(query):
    """Simulated market intelligence database."""
    data = {
        "ev market size": "The global EV market was valued at $388B in 2024, projected to reach $950B by 2030 (CAGR ~16%). China leads with 60% market share.",
        "tesla competitors": "Key Tesla competitors: BYD (largest global EV seller by volume), Volkswagen Group (ID. series), Hyundai-Kia (fastest growing in Europe), and Rivian (US truck segment).",
        "ev margins": "Average EV gross margins: Tesla ~18%, BYD ~20%, legacy OEMs ~5-8% on EVs. Battery costs represent 30-40% of vehicle cost.",
        "southeast asia market": "Southeast Asia EV penetration is ~5% (2024), led by Thailand (govt subsidies) and Indonesia (nickel supply chain). Market expected to grow 25% CAGR through 2030.",
        "mckinsey frameworks": "Key McKinsey frameworks: MECE (Mutually Exclusive, Collectively Exhaustive), 7S Model, Three Horizons, Profit Pools, and Value Chain Analysis."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No data found for: {query}"

def think_node(state: ReActState) -> dict:
    """McKinsey analyst reasons about what data to gather next."""
    ctx = ""
    for i in range(len(state["actions"])):
        ctx += f"Action: {state['actions'][i]}\nObservation: {state['observations'][i]}\n"
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey research analyst. Given the question and any previous research findings, decide your next step. Either search for more data (respond: ACTION: search '<query>') or provide a final synthesized answer (respond: FINAL ANSWER: <answer>). Think like a consultant -- be MECE in your approach."),
        HumanMessage(content=f"Research question: {state['question']}\n\n{ctx}\nWhat should I research next?")
    ])
    thought = response.content
    print(f"  Analyst thinks: {thought[:120]}...")
    return {"thoughts": state["thoughts"] + [thought]}

def act_node(state: ReActState) -> dict:
    """Execute the research action."""
    latest_thought = state["thoughts"][-1]
    
    if "FINAL ANSWER:" in latest_thought:
        answer = latest_thought.split("FINAL ANSWER:")[-1].strip()
        return {"final_answer": answer, "step_count": state["step_count"] + 1}
    
    if "ACTION: search" in latest_thought:
        query = latest_thought.split("search")[-1].strip().strip("'\"")
        observation = research_tool(query)
        print(f"  Research: search('{query}') -> {observation[:80]}...")
        return {
            "actions": state["actions"] + [f"search('{query}')"],
            "observations": state["observations"] + [observation],
            "step_count": state["step_count"] + 1
        }
    
    return {"final_answer": latest_thought, "step_count": state["step_count"] + 1}

def should_continue(state: ReActState) -> str:
    return "end" if state["final_answer"] or state["step_count"] >= 5 else "think"

graph = StateGraph(ReActState)
graph.add_node("think", think_node)
graph.add_node("act", act_node)
graph.set_entry_point("think")
graph.add_edge("think", "act")
graph.add_conditional_edges("act", should_continue, {"think": "think", "end": END})
app = graph.compile()

result = app.invoke({
    "question": "What is the EV market size and who are Tesla's main competitors?",
    "thoughts": [], "actions": [], "observations": [],
    "final_answer": "", "step_count": 0
})
print(f"\nAnalyst's Answer: {result['final_answer']}")
print(f"Research steps taken: {result['step_count']}")

### Demo 4: Cycles for Iterative Refinement -- Refining a Recommendation Until Partner-Quality

In McKinsey, deliverables go through multiple rounds of refinement before they reach partner-quality. This demo models that iterative process: draft a strategic recommendation, have a simulated "partner review" critique it, and loop back to refine until it meets the bar or reaches max iterations.

In [ ]:
# Demo 4 - Cycles: Refining a Recommendation Until Partner-Quality

class RecommendationState(TypedDict):
    strategic_question: str
    draft_recommendation: str
    partner_feedback: str
    iteration: int
    is_partner_quality: bool

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

def formulate_recommendation(state: RecommendationState) -> dict:
    """Draft or refine a strategic recommendation."""
    if state["draft_recommendation"]:
        prompt = f"You are a McKinsey engagement manager. Refine this strategic recommendation based on partner feedback. Make it more MECE, insight-driven, and actionable.\n\nCurrent recommendation: {state['draft_recommendation']}\n\nPartner feedback: {state['partner_feedback']}\n\nProvide an improved recommendation (3-4 sentences, structured and concise):"
    else:
        prompt = f"You are a McKinsey engagement manager. Draft a strategic recommendation for this question (3-4 sentences). Be specific, actionable, and data-driven.\n\nStrategic question: {state['strategic_question']}"
    
    r = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Iteration {state['iteration'] + 1}] Recommendation: {r.content[:100]}...")
    return {"draft_recommendation": r.content, "iteration": state["iteration"] + 1}

def partner_review(state: RecommendationState) -> dict:
    """Simulated partner review -- critiques the recommendation for quality."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner reviewing a strategic recommendation. Rate it 1-10 on: (1) MECE structure, (2) actionability, (3) insight depth. If average is 8+, respond with 'APPROVED - ready for client'. Otherwise, provide specific feedback on what to improve."),
        HumanMessage(content=state["draft_recommendation"])
    ])
    is_approved = "APPROVED" in r.content.upper() or state["iteration"] >= 3
    status = "PARTNER APPROVED" if is_approved else f"Revision needed: {r.content[:80]}"
    print(f"  [Partner Review] {status}...")
    return {"partner_feedback": r.content, "is_partner_quality": is_approved}

def should_refine(state: RecommendationState) -> str:
    return "end" if state["is_partner_quality"] else "formulate_recommendation"

graph = StateGraph(RecommendationState)
graph.add_node("formulate_recommendation", formulate_recommendation)
graph.add_node("partner_review", partner_review)
graph.set_entry_point("formulate_recommendation")
graph.add_edge("formulate_recommendation", "partner_review")
graph.add_conditional_edges("partner_review", should_refine, {"formulate_recommendation": "formulate_recommendation", "end": END})
app = graph.compile()

result = app.invoke({
    "strategic_question": "Should our client, a mid-size European bank, acquire a fintech payments startup to accelerate digital transformation?",
    "draft_recommendation": "", "partner_feedback": "", "iteration": 0, "is_partner_quality": False
})
print(f"\nFinal Recommendation (after {result['iteration']} iterations):")
print(result["draft_recommendation"])

### Demo 5: Human-in-the-Loop -- Partner Sign-Off Before Client Delivery

In consulting, no major deliverable goes to the client without partner approval. This demo models that gate: the system generates a client-ready action plan, pauses for partner review (simulated), and only proceeds to "deliver to client" if approved.

In [ ]:
# Demo 5 - Human-in-the-Loop: Partner Sign-Off Before Client Delivery

class ClientDeliveryState(TypedDict):
    engagement_context: str
    action_plan: str
    partner_approved: bool
    delivery_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def prepare_deliverable(state: ClientDeliveryState) -> dict:
    """Generate a client-ready action plan."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Create a structured 3-step action plan for the client. Include specific timelines, owners, and expected impact for each step."),
        HumanMessage(content=state["engagement_context"])
    ])
    print(f"  Deliverable prepared: {r.content[:120]}...")
    return {"action_plan": r.content}

def partner_signoff_node(state: ClientDeliveryState) -> dict:
    """Simulate partner review gate (auto-approve for demo)."""
    print(f"  [PARTNER REVIEW] Reviewing action plan for client delivery...")
    print(f"    Plan preview: {state['action_plan'][:200]}...")
    # In production, this would pause and wait for actual partner input
    approved = True  # Simulated approval
    print(f"  [PARTNER] {'Approved for client delivery' if approved else 'Needs revision -- send back to team'}")
    return {"partner_approved": approved}

def deliver_to_client(state: ClientDeliveryState) -> dict:
    """Package and deliver the final output to the client."""
    if not state["partner_approved"]:
        return {"delivery_output": "Delivery blocked -- partner approval required."}
    
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey partner delivering recommendations to a C-suite client. Wrap this action plan in a professional executive summary with a compelling opening and clear next steps."),
        HumanMessage(content=f"Action Plan:\n{state['action_plan']}")
    ])
    return {"delivery_output": r.content}

graph = StateGraph(ClientDeliveryState)
graph.add_node("prepare_deliverable", prepare_deliverable)
graph.add_node("partner_signoff", partner_signoff_node)
graph.add_node("deliver_to_client", deliver_to_client)
graph.set_entry_point("prepare_deliverable")
graph.add_edge("prepare_deliverable", "partner_signoff")
graph.add_conditional_edges("partner_signoff", lambda s: "deliver_to_client" if s["partner_approved"] else "end", {
    "deliver_to_client": "deliver_to_client", "end": END
})
graph.add_edge("deliver_to_client", END)
app = graph.compile()

result = app.invoke({
    "engagement_context": "A Fortune 500 CPG company needs to reduce supply chain costs by 15% within 18 months while maintaining service levels",
    "action_plan": "", "partner_approved": False, "delivery_output": ""
})
print(f"\nClient Delivery:\n{result['delivery_output'][:300]}...")

---
## Tasks -- Full Solutions

### Task 1: Build a Client Onboarding Pipeline (Linear Workflow)

**Scenario:** When a new client engagement kicks off, McKinsey runs a standard onboarding pipeline: (1) gather and clean the raw client data, (2) generate a preliminary diagnostic summary using an LLM, (3) translate the diagnostic into an executive-ready brief.

**Approach:** Three sequential nodes: `gather_data` (Python string ops to clean raw input), `analyze_situation` (LLM generates diagnostic summary), `prepare_brief` (LLM translates to executive-ready language). State flows linearly from entry to END.

In [ ]:
# Task 1 - SOLUTION: Client Onboarding Pipeline (Linear Workflow)

class OnboardingState(TypedDict):
    raw_client_data: str
    clean_data: str
    diagnostic_summary: str
    executive_brief: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def gather_data(state: OnboardingState) -> dict:
    """Clean and normalize raw client data."""
    cleaned = " ".join(state["raw_client_data"].split())
    return {"clean_data": cleaned}

def analyze_situation(state: OnboardingState) -> dict:
    """Generate a preliminary diagnostic summary of the client situation."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey associate. Analyze this client data and provide a concise diagnostic summary highlighting key challenges, opportunities, and areas requiring deeper investigation. Use a MECE structure. Keep it to 2-3 sentences."),
        HumanMessage(content=state["clean_data"])
    ])
    return {"diagnostic_summary": r.content}

def prepare_brief(state: OnboardingState) -> dict:
    """Translate the diagnostic into an executive-ready brief."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Transform this diagnostic summary into a polished executive brief suitable for a partner to review before the first client steering committee. Use professional consulting language. Keep it to 2-3 sentences."),
        HumanMessage(content=state["diagnostic_summary"])
    ])
    return {"executive_brief": r.content}

graph = StateGraph(OnboardingState)
graph.add_node("gather_data", gather_data)
graph.add_node("analyze_situation", analyze_situation)
graph.add_node("prepare_brief", prepare_brief)
graph.set_entry_point("gather_data")
graph.add_edge("gather_data", "analyze_situation")
graph.add_edge("analyze_situation", "prepare_brief")
graph.add_edge("prepare_brief", END)
app = graph.compile()

result = app.invoke({
    "raw_client_data": "  Client: GlobalRetail Corp.   Revenue $8.2B,   declining   margins   from 12% to 7%   over 3 years.    Major  cost  pressures  in  supply chain   and  labor.  Considering  digital  transformation   but  lacks  internal  capabilities.   Competitors  gaining  share  in   e-commerce  channel.  ",
    "clean_data": "", "diagnostic_summary": "", "executive_brief": ""
})
print(f"Clean Data: {result['clean_data']}")
print(f"\nDiagnostic: {result['diagnostic_summary']}")
print(f"\nExecutive Brief: {result['executive_brief']}")

### Task 2: PE Due Diligence Router (Conditional Routing)

**Scenario:** A private equity firm hires McKinsey to assess acquisition targets. Each target gets classified by deal size and routed to the appropriate due diligence workstream: **quick screen** (deals < $100M), **standard diligence** ($100M-$1B), or **mega-deal deep dive** (> $1B).

**Approach:** A `classify_deal` node uses the LLM to assess deal size and complexity. Three specialized handler nodes (`quick_screen`, `standard_diligence`, `deep_dive`) each apply different levels of analytical rigor. Conditional edges route based on the classification.

In [ ]:
# Task 2 - SOLUTION: PE Due Diligence Router (Conditional Routing)

class DueDiligenceState(TypedDict):
    deal_description: str
    deal_tier: str
    assessment: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def classify_deal(state: DueDiligenceState) -> dict:
    """Classify the acquisition target by deal size and complexity."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey PE due diligence specialist. Classify this deal into exactly one tier: 'quick_screen' (small bolt-on, < $100M), 'standard' (mid-market, $100M-$1B), or 'deep_dive' (mega-deal, > $1B or highly complex). Respond with just the tier word(s)."),
        HumanMessage(content=state["deal_description"])
    ])
    tier = r.content.strip().lower()
    print(f"  Deal classified as: {tier}")
    return {"deal_tier": tier}

def quick_screen_handler(state: DueDiligenceState) -> dict:
    """Quick 1-week screening for small bolt-on acquisitions."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey associate conducting a rapid deal screen. Provide a concise assessment covering: (1) strategic fit, (2) key financial metrics to verify, (3) go/no-go recommendation. Keep it brief and actionable."),
        HumanMessage(content=state["deal_description"])
    ])
    return {"assessment": f"[QUICK SCREEN - 1 Week]\n{r.content}"}

def standard_diligence_handler(state: DueDiligenceState) -> dict:
    """Standard 4-6 week commercial due diligence."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager leading commercial due diligence. Provide a structured assessment covering: (1) market attractiveness, (2) competitive position, (3) revenue synergies, (4) key risks and mitigants. Use a professional consulting format."),
        HumanMessage(content=state["deal_description"])
    ])
    return {"assessment": f"[STANDARD DILIGENCE - 4-6 Weeks]\n{r.content}"}

def deep_dive_handler(state: DueDiligenceState) -> dict:
    """Comprehensive 8-12 week mega-deal due diligence."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner leading a mega-deal due diligence. Provide a comprehensive assessment covering: (1) market landscape and growth drivers, (2) target's competitive moat, (3) operational improvement levers, (4) synergy quantification framework, (5) integration risks, (6) valuation considerations. Be thorough and executive-ready."),
        HumanMessage(content=state["deal_description"])
    ])
    return {"assessment": f"[DEEP DIVE - 8-12 Weeks]\n{r.content}"}

def route_deal(state: DueDiligenceState) -> str:
    tier = state["deal_tier"]
    if "quick" in tier or "screen" in tier: return "quick_screen"
    if "deep" in tier or "mega" in tier: return "deep_dive"
    return "standard_diligence"

graph = StateGraph(DueDiligenceState)
graph.add_node("classify_deal", classify_deal)
graph.add_node("quick_screen", quick_screen_handler)
graph.add_node("standard_diligence", standard_diligence_handler)
graph.add_node("deep_dive", deep_dive_handler)
graph.set_entry_point("classify_deal")
graph.add_conditional_edges("classify_deal", route_deal, {
    "quick_screen": "quick_screen",
    "standard_diligence": "standard_diligence",
    "deep_dive": "deep_dive"
})
graph.add_edge("quick_screen", END)
graph.add_edge("standard_diligence", END)
graph.add_edge("deep_dive", END)
app = graph.compile()

for deal in [
    "A $50M regional logistics company with 3 warehouses, stable revenues, potential bolt-on for our portfolio company",
    "A $400M B2B SaaS platform in the healthcare space with 20% YoY growth and 85% gross margins",
    "A $5B global specialty chemicals conglomerate with operations in 30 countries, complex carve-out from a larger group"
]:
    r = app.invoke({"deal_description": deal, "deal_tier": "", "assessment": ""})
    print(f"\nDeal: {deal[:80]}...")
    print(f"Assessment: {r['assessment'][:200]}...\n{'='*60}")

### Task 3: MECE Analysis Refinement Loop (Self-Correcting Cycle)

**Scenario:** A core McKinsey discipline is ensuring analyses are MECE (Mutually Exclusive, Collectively Exhaustive). This workflow generates a market analysis, tests it against MECE criteria, and iteratively refines it until the analysis passes quality checks or reaches max attempts.

**Approach:** The `analyze_market` node asks the LLM to produce a structured market analysis. The `assess_quality` node evaluates whether the analysis is MECE. If it fails, the workflow loops back with specific feedback for improvement. A conditional edge loops until quality passes or max attempts (3) is reached.

In [ ]:
# Task 3 - SOLUTION: MECE Analysis Refinement Loop

class MECEAnalysisState(TypedDict):
    market_question: str
    analysis: str
    quality_feedback: str
    is_mece: bool
    attempts: int

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def analyze_market(state: MECEAnalysisState) -> dict:
    """Generate or refine a MECE market analysis."""
    if state["analysis"] and state["quality_feedback"]:
        prompt = f"You are a McKinsey associate. Revise this market analysis to address the quality feedback. Ensure the analysis is MECE (Mutually Exclusive, Collectively Exhaustive).\n\nCurrent analysis:\n{state['analysis']}\n\nFeedback:\n{state['quality_feedback']}\n\nProvide the improved analysis with clearly structured, non-overlapping categories that cover the full market."
    else:
        prompt = f"You are a McKinsey associate. Provide a structured market analysis for: {state['market_question']}\n\nStructure your analysis in MECE segments (Mutually Exclusive, Collectively Exhaustive). Use 3-4 distinct segments with brief descriptions and estimated market shares."
    
    r = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Attempt {state['attempts'] + 1}] Analysis: {r.content[:100]}...")
    return {"analysis": r.content, "attempts": state["attempts"] + 1}

def assess_quality(state: MECEAnalysisState) -> dict:
    """Evaluate whether the analysis meets MECE quality standards."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager reviewing an analysis for MECE quality. Check: (1) Are segments mutually exclusive (no overlaps)? (2) Are segments collectively exhaustive (nothing missing)? (3) Is the logic clean and the data plausible? If all criteria are met, respond with 'MECE APPROVED'. Otherwise, provide specific feedback on what overlaps or gaps exist."),
        HumanMessage(content=state["analysis"])
    ])
    is_mece = "APPROVED" in r.content.upper() or state["attempts"] >= 3
    print(f"  [Quality Check] {'MECE APPROVED' if is_mece else r.content[:80]}...")
    return {"quality_feedback": r.content, "is_mece": is_mece}

def check_quality(state: MECEAnalysisState) -> str:
    if state["is_mece"] or state["attempts"] >= 3:
        return "end"
    return "analyze_market"

graph = StateGraph(MECEAnalysisState)
graph.add_node("analyze_market", analyze_market)
graph.add_node("assess_quality", assess_quality)
graph.set_entry_point("analyze_market")
graph.add_edge("analyze_market", "assess_quality")
graph.add_conditional_edges("assess_quality", check_quality, {"analyze_market": "analyze_market", "end": END})
app = graph.compile()

result = app.invoke({
    "market_question": "Segment the global cloud computing market for a client considering entry",
    "analysis": "", "quality_feedback": "", "is_mece": False, "attempts": 0
})
print(f"\nFinal MECE Analysis (after {result['attempts']} attempts):")
print(result["analysis"])
print(f"\nMECE Approved: {result['is_mece']}")

### Task 4: Market Entry Assessment Agent (Planning and Execution)

**Scenario:** A client wants to enter a new geographic market. McKinsey deploys a structured approach: first plan the assessment (identify key research areas), then execute each research step systematically, and finally synthesize findings into an actionable market entry recommendation.

**Approach:** The `scope_assessment` node generates a list of research workstreams. The `execute_workstream` node processes one workstream per invocation. The `check_progress` node routes back to execute if more workstreams remain. Finally, the `synthesize_recommendation` node combines all findings into a market entry recommendation.

In [ ]:
# Task 4 - SOLUTION: Market Entry Assessment Agent

class MarketEntryState(TypedDict):
    market_entry_question: str
    workstreams: list[str]
    current_workstream: int
    findings: list[str]
    recommendation: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def scope_assessment(state: MarketEntryState) -> dict:
    """Plan the market entry assessment with key research workstreams."""
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager scoping a market entry assessment. Define exactly 3 research workstreams as a JSON list of strings. Each workstream should be a specific, actionable research area (e.g., 'Assess market size and growth trajectory', 'Map competitive landscape and identify white spaces', 'Evaluate regulatory environment and entry barriers'). Return ONLY the JSON list."),
        HumanMessage(content=f"Market entry question: {state['market_entry_question']}")
    ])
    try:
        workstreams = json.loads(r.content)
    except:
        workstreams = ["Assess market size and growth drivers", "Analyze competitive landscape and positioning", "Evaluate entry barriers and go-to-market options"]
    print(f"  Workstreams scoped: {workstreams}")
    return {"workstreams": workstreams}

def execute_workstream(state: MarketEntryState) -> dict:
    """Execute the current research workstream."""
    workstream = state["workstreams"][state["current_workstream"]]
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey research analyst executing a workstream for a market entry assessment. Provide a concise, data-informed finding (2-3 sentences) with specific metrics or insights where possible. Think like a consultant -- be specific and actionable."),
        HumanMessage(content=f"Market entry context: {state['market_entry_question']}\nWorkstream: {workstream}")
    ])
    print(f"  Workstream {state['current_workstream'] + 1}: {r.content[:100]}...")
    return {"findings": state["findings"] + [r.content], "current_workstream": state["current_workstream"] + 1}

def check_progress(state: MarketEntryState) -> str:
    """Check if all workstreams are complete."""
    if state["current_workstream"] >= len(state["workstreams"]):
        return "synthesize_recommendation"
    return "execute_workstream"

def synthesize_recommendation(state: MarketEntryState) -> dict:
    """Synthesize all workstream findings into a market entry recommendation."""
    findings_text = "\n".join(f"Workstream {i+1}: {f}" for i, f in enumerate(state["findings"]))
    r = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner synthesizing research findings into an actionable market entry recommendation. Provide a clear go/no-go recommendation with supporting rationale, key success factors, and recommended entry approach. Use a professional consulting tone."),
        HumanMessage(content=f"Market entry question: {state['market_entry_question']}\n\nResearch Findings:\n{findings_text}")
    ])
    return {"recommendation": r.content}

graph = StateGraph(MarketEntryState)
graph.add_node("scope_assessment", scope_assessment)
graph.add_node("execute_workstream", execute_workstream)
graph.add_node("check_progress", check_progress)
graph.add_node("synthesize_recommendation", synthesize_recommendation)
graph.set_entry_point("scope_assessment")
graph.add_edge("scope_assessment", "execute_workstream")
graph.add_edge("execute_workstream", "check_progress")
graph.add_conditional_edges("check_progress", check_progress, {
    "execute_workstream": "execute_workstream",
    "synthesize_recommendation": "synthesize_recommendation"
})
graph.add_edge("synthesize_recommendation", END)
app = graph.compile()

result = app.invoke({
    "market_entry_question": "Should a US-based digital health platform expand into the Indian healthcare market?",
    "workstreams": [], "current_workstream": 0, "findings": [], "recommendation": ""
})
print(f"\nMarket Entry Recommendation:\n{result['recommendation']}")

---
## Summary

In this session students learned LangGraph workflow orchestration through McKinsey consulting scenarios:

1. **StateGraph Basics** -- Typed state, nodes as functions, edges for control flow. Modeled as a consulting engagement pipeline (scope, classify, brief).
2. **Conditional Routing** -- Dynamic branching based on LLM decisions. Modeled as engagement complexity routing (rapid assessment vs. standard vs. transformation).
3. **ReAct Agents** -- The Reason-Act-Observe loop as a cyclic graph. Modeled as a McKinsey market research analyst gathering competitive intelligence.
4. **Iterative Refinement** -- Self-correcting workflows with write-critique-improve cycles. Modeled as refining recommendations until partner-quality.
5. **Human-in-the-Loop** -- Pause points for human oversight. Modeled as partner sign-off before client delivery.

**Instructor Notes:**
- Emphasize that LangGraph makes control flow *explicit* -- students can draw the graph on a whiteboard, just like a McKinsey process map.
- For Task 3 (MECE refinement), discuss how this pattern mirrors the real consulting quality bar -- deliverables iterate until they meet partner standards.
- For Task 4 (market entry), note how the scope-execute-check-synthesize pattern generalizes to any structured consulting assessment.
- Connect to real McKinsey work: the patterns here (conditional routing by client tier, iterative refinement, human gates) are exactly how production consulting AI tools would be designed.

**Up Next:** In Session 4, multi-agent architectures -- supervisor-worker, handoffs, and collaborative systems (modeled as McKinsey team structures: partner-EM-associate hierarchies).